In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from shapely.geometry import Point, Polygon
from scipy.spatial import ConvexHull


In [5]:
#excel_file_path1 = 'C:/Users/ASUS/OneDrive/Documents/中信兄弟/Season_data/2024_non_trackman.xlsx'
#excel_file_path2 = 'C:/Users/ASUS/OneDrive/Documents/中信兄弟/Season_data/2023_non_trackman.xlsx'
excel_file_path3 = 'C:/Users/ASUS/OneDrive/Documents/中信兄弟/Season_data/2025_non_trackman.xlsx'
sheet_name = 'Sheet1'

In [6]:
#data2024 = pd.read_excel(excel_file_path1, sheet_name=sheet_name)
#data2023 = pd.read_excel(excel_file_path2, sheet_name=sheet_name)
data2025 = pd.read_excel(excel_file_path3, sheet_name=sheet_name)

In [7]:
col = ['umpireInChief','date','stadium','bHand','pHand','strikes','balls','side','inning','coordX','coordY','pType','call','result']

In [8]:
#data2024= data2024[col]
#data2023= data2023[col]
df= data2025[col]

In [9]:
#data = pd.concat([data2024, data2025], ignore_index=True)

In [10]:
df.loc[:, 'Year'] = df['date'].str[:4]

C:\Users\ASUS\AppData\Local\Temp\ipykernel_17272\3836225080.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:, 'Year'] = df['date'].str[:4]


In [11]:
df = df[~df['result'].isin(['GO','2B', '1B', 'FO', 'SH', 'E', 'GIDP', 'FC','HBP', 'HR', '3B', 'IBB', 'SF', 'IGNORE', 'DP', 'D3S', 'ID', 'IH'])]
df = df[~df['call'].isin(['F', 'SW', 'CS', 'FT_MISS', 'FOUL_BUNT','FT','TRY_BUNT', 'H', 'BUNT'])]
df= df.dropna(subset=['call'])

In [12]:
name_mapping = {
    'FF':'直球', 
    'CH':'變速球', 
    'FC':'卡特球', 
    'CU':'曲球', 
    'FO':'指叉球',
    'SL':'滑球', 
    'SI':'伸卡球',
    'KN':'彈指曲球', 
    'EP':'小便球'
}

# 新增一列"球員"，根据"Pitcher"列的值匹配中文名
df['球種'] = df['pType'].map(name_mapping)

In [13]:
feild_items = df['stadium'].unique()
feild_items

array(['澄清湖棒球場', '臺中洲際棒球場 Taichung Intercontinental Baseball Stadium',
       '新北市立新莊棒球場', '澄清湖棒球場 Chengcing Lake Baseball Stadium', '嘉義市立棒球場',
       '臺北市立天母棒球場', nan, '新北市立新莊棒球場 Xinzhuang Baseball Stadium',
       '樂天桃園棒球場', '樂天桃園棒球場 Rakuten Taoyuan Baseball Stadium', '臺北大巨蛋',
       '臺北大巨蛋 Taipei Dome', '臺東棒球村第一棒球場', '斗六棒球場',
       '臺北市立天母棒球場 Tianmu Baseball Stadium',
       '臺南市立棒球場 Tainan Municipal Baseball Stadium',
       '嘉義市立棒球場 Chiayi Baseball Field', '花蓮縣立德興棒球場'], dtype=object)

In [14]:
# 判斷投球是否在規則書好球帶內
def in_strike_zone(x, y):
    return -50 <= x <= 50 and -50 <= y <= 50

df["rulebook_strike"] = df.apply(lambda row: in_strike_zone(row["coordX"], row["coordY"]), axis=1)

In [15]:
df

,umpireInChief,date,stadium,bHand,pHand,strikes,balls,side,inning,coordX,coordY,pType,call,result,Year,球種,rulebook_strike
0,林金達,2025-10-07T05:59:47.776Z,澄清湖棒球場,L,L,0,0,AWAY,1,-46.96,-35.34,SI,S,NaN,2025,伸卡球,True
1,林金達,2025-10-07T05:59:47.776Z,澄清湖棒球場,L,L,1,0,AWAY,1,36.10,30.59,SI,S,NaN,2025,伸卡球,True
2,林金達,2025-10-07T05:59:47.776Z,澄清湖棒球場,L,L,2,0,AWAY,1,69.06,70.14,SI,B,NaN,2025,伸卡球,False
4,林金達,2025-10-07T05:59:47.776Z,澄清湖棒球場,L,L,0,0,AWAY,1,-19.27,64.87,SI,S,NaN,2025,伸卡球,False
7,林金達,2025-10-07T05:59:47.776Z,澄清湖棒球場,L,L,2,0,AWAY,1,160.04,-44.57,SL,B,NaN,2025,滑球,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105430,張展榮,2025-03-29T09:00:00.000Z,臺北大巨蛋 Taipei Dome,L,R,1,1,HOME,9,64.29,-15.70,FF,B,NaN,2025,直球,False
105434,張展榮,2025-03-29T09:00:00.000Z,臺北大巨蛋 Taipei Dome,L,R,0,0,HOME,9,7.36,99.72,FF,B,NaN,2025,直球,False
105436,張展榮,2025-03-29T09:00:00.000Z,臺北大巨蛋 Taipei Dome,R,R,0,0,HOME,9,-56.09,-133.56,SL,B,NaN,2025,滑球,False
105437,張展榮,2025-03-29T09:00:00.000Z,臺北大巨蛋 Taipei Dome,R,R,0,1,HOME,9,-54.30,-42.98,FF,S,NaN,2025,直球,False


In [16]:
df["is_correct"] = df.apply(
    lambda row: (row["call"] == "S" and row["rulebook_strike"]) or 
                (row["call"] == "B" and not row["rulebook_strike"]),
    axis=1
)

accuracy_by_umpire = df.groupby("umpireInChief")["is_correct"].mean().reset_index()
accuracy_by_umpire.rename(columns={"is_correct":"accuracy"}, inplace=True)
accuracy_by_umpire['accuracy']=accuracy_by_umpire['accuracy']*100
accuracy_by_umpire

,umpireInChief,accuracy
0,劉世偉,89.342629
1,吳家維,89.584488
2,尤志欽,89.661979
3,張展榮,89.014414
4,彭楚雲,89.815070
5,林金達,88.092156
6,楊崇煇,90.673176
7,江春緯,88.405316
8,王俊宏,87.696904
9,紀華文,89.700000


## TM場次 準確率

In [17]:
stadiums_of_interest = [
    '臺北市立天母棒球場 Tianmu Baseball Stadium',
    '臺中洲際棒球場 \tTaichung Intercontinental Baseball Stadium',
    '臺中洲際棒球場',
    '斗六棒球場 \tDouliu Baseball Stadium',
    '臺北市立天母棒球場',
    '斗六棒球場',
    '臺北大巨蛋 Taipei Dome',
    '臺北大巨蛋',
    '新北市立新莊棒球場 Xinzhuang Baseball Stadium',
    '新北市立新莊棒球場'
    ]

# 使用 loc 方法筛选数据
df2 = df[df['stadium'].isin(stadiums_of_interest)]
df2["is_correct"] = df2.apply(
    lambda row: (row["call"] == "S" and row["rulebook_strike"]) or 
                (row["call"] == "B" and not row["rulebook_strike"]),
    axis=1
)

accuracy_by_umpire2 = df2.groupby("umpireInChief")["is_correct"].mean().reset_index()
accuracy_by_umpire2.rename(columns={"is_correct":"accuracy"}, inplace=True)
accuracy_by_umpire['accuracy']=accuracy_by_umpire['accuracy']*100
accuracy_by_umpire2


C:\Users\ASUS\AppData\Local\Temp\ipykernel_17272\2541566500.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2["is_correct"] = df2.apply(


,umpireInChief,accuracy
0,劉世偉,0.896203
1,吳家維,0.911784
2,尤志欽,0.909953
3,張展榮,0.911497
4,彭楚雲,0.907158
5,林金達,0.888406
6,楊崇煇,0.900546
7,江春緯,0.883591
8,王俊宏,0.887173
9,紀華文,0.899823


# 一致性計算 繪圖

## 1.建立好球帶 (IRn)

In [15]:
df_ump = df

In [105]:
#論文 bandwidth=18, threshold=0.55
#一般繪圖 bandwidth=12, threshold=0.35

from shapely.geometry import Point, Polygon
import numpy as np
from scipy.stats import gaussian_kde

def build_umpire_zone(df_ump, bandwidth=12, threshold=0.35, grid_size=300, outlier_std=3.0):
    """
    用 Kernel Density Estimation (KDE) 建立裁判的實際好球帶區域。
    
    增強功能：
        🔹 自動刪除離群好球點（超過 outlier_std × 標準差）

    參數：
        df_ump : DataFrame（單一裁判＋單一年分＋單一打擊方向）
        bandwidth : 平滑參數，越大越平滑
        threshold : 密度門檻（0~1）
        grid_size : 計算格點數
        outlier_std : 移除超出此倍標準差的異常點（預設 3.0）

    回傳：
        Polygon（KDE strike zone），若資料太少或無效則回傳 None
    """
    strikes = df_ump[df_ump["call"] == "S"][["coordX", "coordY"]].values
    if len(strikes) < 20:
        return None

    # ---------------------------------
    # Step 1️⃣ 移除離群值（outliers）
    # ---------------------------------
    mean = np.mean(strikes, axis=0)
    std = np.std(strikes, axis=0)
    z_score = np.abs((strikes - mean) / std)
    mask = (z_score < outlier_std).all(axis=1)
    strikes = strikes[mask]

    if len(strikes) < 10:
        return None  # 過濾後資料太少

    # ---------------------------------
    # Step 2️⃣ 建立 KDE 模型
    # ---------------------------------
    kde = gaussian_kde(strikes.T, bw_method=bandwidth / np.std(strikes, axis=0).mean())

    # 建立格點區域
    x_min, x_max = strikes[:,0].min() - 20, strikes[:,0].max() + 20
    y_min, y_max = strikes[:,1].min() - 20, strikes[:,1].max() + 20
    x, y = np.meshgrid(np.linspace(x_min, x_max, grid_size),
                       np.linspace(y_min, y_max, grid_size))
    xy = np.vstack([x.ravel(), y.ravel()])
    z = kde(xy).reshape(grid_size, grid_size)

    # ---------------------------------
    # Step 3️⃣ 轉換為多邊形區域
    # ---------------------------------
    z = z / z.max()
    mask = z >= threshold
    pts = np.column_stack([x[mask], y[mask]])

    if len(pts) < 3:
        return None

    from scipy.spatial import ConvexHull
    hull = ConvexHull(pts)
    zone = Polygon(pts[hull.vertices])

    if not zone.is_valid:
        return None
    return zone



## 2.建立標準好球帶（rulebook zone）

In [112]:
true_zone = Polygon([
    (-50, -50),
    (-50, 50),
    (50, 50),
    (50, -50)
])

## 3.標記判決是否正確 + 一致 

In [113]:
def evaluate_calls(df_ump, ump_zone, true_zone):
    """
    評估裁判每顆球的正確性與一致性。
    同時計算錯誤類型。
    """
    results = []
    for _, row in df_ump.iterrows():
        pt = Point(row["coordX"], row["coordY"])
        inside_true = true_zone.contains(pt)
        inside_ump = ump_zone.contains(pt) if ump_zone else False

        # 正確性 (Accuracy)
        correct = (inside_true and row["call"] == "S") or (not inside_true and row["call"] == "B")

        # 一致性 (Consistency)
        consistent = (inside_ump and row["call"] == "S") or (not inside_ump and row["call"] == "B")

        # 錯誤類型
        if inside_true and row["call"] == "B":
            miss_type = "TrueStrike_CalledBall"
        elif (not inside_true) and row["call"] == "S":
            miss_type = "TrueBall_CalledStrike"
        else:
            miss_type = "Correct"

        results.append((correct, consistent, miss_type))

    df_ump[["Correct", "Consistent", "MissType"]] = results
    return df_ump


## 4.計算 Accuracy / Consistency

In [114]:
def compute_metrics(df_ump):
    """
    計算 Accuracy / Consistency。
    並根據「2好球前 / 2好球後」做分組。
    """
    overall_accuracy = df_ump["Correct"].mean()
    overall_consistency = df_ump["Consistent"].mean()

    # 分組統計：2好球後 vs 2好球前
    group_conditions = [
        ('2好球後', df_ump[df_ump['strikes'] == 2]),
        ('2好球前', df_ump[df_ump['strikes'] != 2])
    ]

    subgroup_results = {}
    for label, sub_df in group_conditions:
        if len(sub_df) > 0:
            acc = sub_df["Correct"].mean()
            con = sub_df["Consistent"].mean()
            subgroup_results[label] = {"Accuracy": acc, "Consistency": con}
        else:
            subgroup_results[label] = {"Accuracy": np.nan, "Consistency": np.nan}

    return overall_accuracy, overall_consistency, subgroup_results

## 5.視覺化 (含百分比文字)

In [115]:
import numpy as np
import os
import matplotlib.pyplot as plt
from shapely.geometry import Point

# -----------------------------
# 視覺化函式
# -----------------------------
def plot_umpire(df_ump, ump_zone, umpire, year, bHand, acc, con, subgroup_label, n=10):
    fig, ax = plt.subplots(figsize=(6, 6))

    # --- Rulebook Zone ---
    x, y = true_zone.exterior.xy
    ax.plot(x, y, "k-", lw=2, label="Rulebook Zone")

    # --- Umpire Zone ---
    if ump_zone and not ump_zone.is_empty:
        x, y = ump_zone.exterior.xy
        ax.plot(x, y, color="orange", ls="--", lw=2.5, label="Zone (KDE)")

    # --- Missed Calls ---
    df_miss = df_ump[df_ump["MissType"].isin(["TrueStrike_CalledBall", "TrueBall_CalledStrike"])]
    if not df_miss.empty:
        colors = {
            "TrueStrike_CalledBall": "red",
            "TrueBall_CalledStrike": "green"
        }
        ax.scatter(df_miss["coordX"], df_miss["coordY"],
                   c=df_miss["MissType"].map(colors),
                   alpha=0.9, edgecolors="k", s=35, linewidth=0.3)

    # --- Title & Summary Text ---
    ax.set_title(f"{umpire} ({year}, {bHand}, {subgroup_label})", fontsize=14, weight="bold")
    ax.text(0, 200,
            f"Accuracy: {acc*100:.1f}%   Consistency: {con*100:.1f}%",
            ha="center", va="center", fontsize=12, weight="bold",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.8))

    ax.set_xlabel("coordX")
    ax.set_ylabel("coordY")
    ax.legend()
    ax.grid(True)
    ax.set_xlim(-150, 150)
    ax.set_ylim(-150, 150)
    ax.set_aspect('equal', adjustable='box')

    return fig, ax



## 6. 輸出圖

In [116]:
n = 10
plt.rcParams['font.sans-serif'] = ['Taipei Sans TC', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

output_root = "Umpire_Zones"
os.makedirs(output_root, exist_ok=True)

for (umpire, year, bHand), df_ump in df.groupby(["umpireInChief", "Year", "bHand"]):

    # 清理資料
    df_ump = df_ump.replace([np.inf, -np.inf], np.nan).dropna(subset=["coordX", "coordY"])
    if df_ump.empty:
        print(f"⚠️ {umpire} ({year}, {bHand}) 資料不足或全為 NaN，略過。")
        continue

    # 裁判 KDE 好球帶
    ump_zone = build_umpire_zone(df_ump)
    if ump_zone is None:
        print(f"⚠️ {umpire} ({year}, {bHand}) 無法建立好球帶，略過。")
        continue

    # 評估判決
    df_ump = evaluate_calls(df_ump, ump_zone, true_zone)

    # 計算總體 + 子群
    overall_acc, overall_con, group_stats = compute_metrics(df_ump)

    # 建立輸出資料夾
    save_dir = os.path.join(output_root, f"{umpire}_{year}")
    os.makedirs(save_dir, exist_ok=True)

    # 🔹 依「2好球前 / 2好球後」畫圖
    for label, sub_df in [
        ("2好球前", df_ump[df_ump["strikes"] != 2]),
        ("2好球後", df_ump[df_ump["strikes"] == 2])
    ]:
        if sub_df.empty:
            print(f"⚠️ {umpire} ({year}, {bHand}, {label}) 無資料，略過。")
            continue

        acc = group_stats[label]["Accuracy"]
        con = group_stats[label]["Consistency"]

        # 繪圖
        fig, ax = plot_umpire(sub_df, ump_zone, umpire, year, bHand, acc, con, label, n)

        # 儲存
        save_path = os.path.join(save_dir, f"{bHand}_{label}_zone.png")
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close(fig)

        print(f"✅ 已儲存：{save_path}")


✅ 已儲存：Umpire_Zones\劉世偉_2025\L_2好球前_zone.png
✅ 已儲存：Umpire_Zones\劉世偉_2025\L_2好球後_zone.png
✅ 已儲存：Umpire_Zones\劉世偉_2025\R_2好球前_zone.png
✅ 已儲存：Umpire_Zones\劉世偉_2025\R_2好球後_zone.png
✅ 已儲存：Umpire_Zones\吳家維_2025\L_2好球前_zone.png
✅ 已儲存：Umpire_Zones\吳家維_2025\L_2好球後_zone.png
✅ 已儲存：Umpire_Zones\吳家維_2025\R_2好球前_zone.png
✅ 已儲存：Umpire_Zones\吳家維_2025\R_2好球後_zone.png
✅ 已儲存：Umpire_Zones\尤志欽_2025\L_2好球前_zone.png
✅ 已儲存：Umpire_Zones\尤志欽_2025\L_2好球後_zone.png
✅ 已儲存：Umpire_Zones\尤志欽_2025\R_2好球前_zone.png
✅ 已儲存：Umpire_Zones\尤志欽_2025\R_2好球後_zone.png
✅ 已儲存：Umpire_Zones\張展榮_2025\L_2好球前_zone.png
✅ 已儲存：Umpire_Zones\張展榮_2025\L_2好球後_zone.png
✅ 已儲存：Umpire_Zones\張展榮_2025\R_2好球前_zone.png
✅ 已儲存：Umpire_Zones\張展榮_2025\R_2好球後_zone.png
✅ 已儲存：Umpire_Zones\彭楚雲_2025\L_2好球前_zone.png
✅ 已儲存：Umpire_Zones\彭楚雲_2025\L_2好球後_zone.png
✅ 已儲存：Umpire_Zones\彭楚雲_2025\R_2好球前_zone.png
✅ 已儲存：Umpire_Zones\彭楚雲_2025\R_2好球後_zone.png
✅ 已儲存：Umpire_Zones\林金達_2025\L_2好球前_zone.png
✅ 已儲存：Umpire_Zones\林金達_2025\L_2好球後_zone.png
✅ 已儲存：Umpire_Zones\林金達_2025\R_2好